# Neural Network Image Classification
**Author:** Susai Florian

Skin Moles ("Vai de pielea mea") + Facial Expressions ("You're on candid camera")

This notebook implements and evaluates MLP, CNN, and fine-tuned ResNet-18 architectures on two image classification tasks, with systematic ablation studies.

## Section 0: Imports
All required libraries: PyTorch, torchvision, scikit-learn, matplotlib.

In [ ]:
import os
import csv
import json
import random
import copy
import numpy as np
import pandas as pd
from collections import Counter, OrderedDict
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)


## Section 1: Configuration
Hyperparameters, dataset paths, class definitions, normalization statistics.

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()

SKIN_DIR = os.path.join(BASE_DIR, "vai-de-pielea-mea")
FACE_DIR = os.path.join(BASE_DIR, "you-re-on-candid-camera")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
NUM_WORKERS = 8

# Output directories
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
SUB_DIR = os.path.join(RESULTS_DIR, "submissions")
RUNS_DIR = os.path.join(BASE_DIR, "runs")

for d in [RESULTS_DIR, FIG_DIR, SUB_DIR, RUNS_DIR]:
    os.makedirs(d, exist_ok=True)

# Class definitions
SKIN_CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
SKIN_CLASS_NAMES = [
    'Bowel disease', 'Basal Cell Carcinoma', 'Benign Keratosis',
    'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesion'
]
FACE_CLASSES = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']

# Normalization stats
SKIN_MEAN = [0.7646, 0.5464, 0.5712]
SKIN_STD  = [0.1410, 0.1532, 0.1706]

with open(os.path.join(FACE_DIR, "splits", "norm_stats.json")) as f:
    face_stats = json.load(f)
FACE_MEAN = face_stats["mean"]
FACE_STD  = face_stats["std"]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training hyperparameters
MLP_IMG_SIZE = 64
CNN_IMG_SIZE = 96
FT_IMG_SIZE  = 224

MLP_EPOCHS = 20
CNN_EPOCHS = 25
FT_EPOCHS  = 20

MLP_BS = 256
CNN_BS = 128
FT_BS  = 32

MLP_LR = 1e-3
CNN_LR = 1e-3
FT_LR_HEAD = 1e-3
FT_LR_BACKBONE = 1e-4

WEIGHT_DECAY = 1e-4

print(f"Device: {DEVICE}")
print(f"Base dir: {BASE_DIR}")


## Section 2: Seed and Utilities
Reproducibility setup with global seed.

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed()


## Section 3: Dataset Classes
Custom PyTorch Dataset classes for skin moles and facial expressions.

In [ ]:
class SkinDataset(Dataset):
    """Dataset for skin moles (Vai de pielea mea)."""
    def __init__(self, csv_path, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.samples = []  # (filename, label_idx)
        self.class_to_idx = {c: i for i, c in enumerate(SKIN_CLASSES)}

        with open(csv_path, newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                label = self.class_to_idx[row['diagnostic']]
                self.samples.append((row['imagine'], label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, label = self.samples[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_labels(self):
        return np.array([s[1] for s in self.samples])

    def get_image_path(self, idx):
        return os.path.join(self.img_dir, self.samples[idx][0])


class SkinTestDataset(Dataset):
    """Kaggle test dataset for skin moles (no labels)."""
    def __init__(self, csv_path, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.images = []

        with open(csv_path, newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.images.append(row['imagine'])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        fname = self.images[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, fname


class FaceDataset(Dataset):
    """Dataset for facial expressions (You're on candid camera)."""
    def __init__(self, csv_path, base_dir, transform=None):
        self.base_dir = base_dir
        self.transform = transform
        self.samples = []  # (full_path, label_idx)

        with open(csv_path, newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                path = os.path.join(base_dir, row['path'])
                label = int(row['label']) - 1  # Convert 1-7 to 0-6
                self.samples.append((path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_labels(self):
        return np.array([s[1] for s in self.samples])

    def get_image_path(self, idx):
        return self.samples[idx][0]


class FaceRemoteTestDataset(Dataset):
    """Kaggle remote test dataset for faces (no labels)."""
    def __init__(self, csv_path, base_dir, transform=None):
        self.transform = transform
        self.ids = []
        self.path_map = {}

        # Build mapping from filename to full path
        test_img_dir = os.path.join(base_dir, 'DATASET', 'test')
        for class_dir in sorted(os.listdir(test_img_dir)):
            class_path = os.path.join(test_img_dir, class_dir)
            if os.path.isdir(class_path):
                for fname in os.listdir(class_path):
                    self.path_map[fname] = os.path.join(class_path, fname)

        with open(csv_path, newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.ids.append(row['id'])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        fname = self.ids[idx]
        img = Image.open(self.path_map[fname]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, fname


## Section 4: Transform Pipelines
Data augmentation and normalization transforms: none, geometric, color, all.

In [ ]:
def get_norm_stats(dataset_name, for_pretrained=False):
    if for_pretrained:
        return IMAGENET_MEAN, IMAGENET_STD
    return (SKIN_MEAN, SKIN_STD) if dataset_name == 'skin' else (FACE_MEAN, FACE_STD)


def get_transforms(dataset_name, img_size, aug_config='none', for_pretrained=False):
    """Build train and val transforms.

    aug_config: 'none', 'geometric', 'color', 'all'
    """
    mean, std = get_norm_stats(dataset_name, for_pretrained)

    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    if aug_config == 'none':
        return val_tf, val_tf

    train_list = []

    # Geometric augmentations
    if aug_config in ('geometric', 'all'):
        bigger = int(img_size * 1.15)
        train_list.append(transforms.Resize((bigger, bigger)))
        train_list.append(transforms.RandomCrop(img_size))
        train_list.append(transforms.RandomHorizontalFlip(p=0.5))
        if dataset_name == 'skin':
            train_list.append(transforms.RandomVerticalFlip(p=0.5))
            train_list.append(transforms.RandomRotation(180))
        else:
            train_list.append(transforms.RandomRotation(15))
    else:
        train_list.append(transforms.Resize((img_size, img_size)))

    # Color augmentations
    if aug_config in ('color', 'all'):
        if dataset_name == 'skin':
            train_list.append(transforms.ColorJitter(brightness=0.15, contrast=0.15,
                                                     saturation=0.1, hue=0.05))
        else:
            train_list.append(transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                                     saturation=0.2, hue=0.1))
        train_list.append(transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)))

    train_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_tf = transforms.Compose(train_list)
    return train_tf, val_tf


## Section 5: Model Definitions
MLP (3 hidden layers with BN+ReLU+Dropout), CNN (4 conv blocks + GAP + 2 FC), and FocalLoss.

In [ ]:
class MLP(nn.Module):
    """Multi-Layer Perceptron for image classification.

    Architecture: Flatten -> [Linear -> BN1d -> ReLU -> (Dropout)] x 3 -> Linear
    """
    def __init__(self, input_dim, num_classes, use_dropout=True, dropout_p=0.3):
        super().__init__()
        layers = [nn.Flatten()]

        # Layer 1: input_dim -> 512
        layers.append(nn.Linear(input_dim, 512))
        layers.append(nn.BatchNorm1d(512))
        layers.append(nn.ReLU(inplace=True))
        if use_dropout:
            layers.append(nn.Dropout(dropout_p))

        # Layer 2: 512 -> 256
        layers.append(nn.Linear(512, 256))
        layers.append(nn.BatchNorm1d(256))
        layers.append(nn.ReLU(inplace=True))
        if use_dropout:
            layers.append(nn.Dropout(dropout_p))

        # Layer 3: 256 -> 128
        layers.append(nn.Linear(256, 128))
        layers.append(nn.BatchNorm1d(128))
        layers.append(nn.ReLU(inplace=True))
        if use_dropout:
            layers.append(nn.Dropout(dropout_p))

        # Output layer
        layers.append(nn.Linear(128, num_classes))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    """CNN for image classification.

    Architecture: 4 conv blocks + AdaptiveAvgPool + 2 FC layers.
    Constraints: max 4 conv layers, max 2 FC layers.
    """
    def __init__(self, num_classes, first_kernel=3, use_batchnorm=True, use_dropout=True):
        super().__init__()
        self.use_batchnorm = use_batchnorm

        def conv_block(in_ch, out_ch, kernel_size=3, pool=True):
            padding = kernel_size // 2
            block = [nn.Conv2d(in_ch, out_ch, kernel_size, padding=padding,
                               bias=not use_batchnorm)]
            if use_batchnorm:
                block.append(nn.BatchNorm2d(out_ch))
            block.append(nn.ReLU(inplace=True))
            if pool:
                block.append(nn.MaxPool2d(2))
            return block

        features = []
        # Block 1
        features.extend(conv_block(3, 32, kernel_size=first_kernel))
        # Block 2
        features.extend(conv_block(32, 64))
        # Block 3
        features.extend(conv_block(64, 128))
        # Block 4
        features.extend(conv_block(128, 256, pool=False))
        features.append(nn.AdaptiveAvgPool2d(1))  # Global Average Pooling

        self.features = nn.Sequential(*features)

        clf_layers = []
        clf_layers.append(nn.Linear(256, 128))
        if use_batchnorm:
            clf_layers.append(nn.BatchNorm1d(128))
        clf_layers.append(nn.ReLU(inplace=True))
        if use_dropout:
            clf_layers.append(nn.Dropout(0.5))
        clf_layers.append(nn.Linear(128, num_classes))

        self.classifier = nn.Sequential(*clf_layers)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance."""
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        p_t = torch.exp(-ce)
        loss = (1 - p_t) ** self.gamma * ce
        if self.reduction == "mean":
            return loss.mean()
        return loss.sum() if self.reduction == "sum" else loss


## Section 6: Training and Evaluation
Training loop, evaluation, class weight computation, weighted sampler, and experiment runner.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = correct / total
    f1_mac = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return running_loss / total, acc, f1_mac


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = correct / total
    f1_mac = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_mic = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    return running_loss / total, acc, f1_mac, f1_mic, np.array(all_preds), np.array(all_labels)


def freeze_bn(model):
    """Freeze BatchNorm layers (use running stats, stop updating gamma/beta)."""
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False


def get_class_weights(labels, num_classes, device):
    """Compute inverse-frequency class weights."""
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def get_weighted_sampler(labels):
    """Create WeightedRandomSampler for balanced batches."""
    counts = np.bincount(labels)
    class_weights = 1.0 / counts.astype(np.float64)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).double(),
        num_samples=len(sample_weights),
        replacement=True,
    )


def run_experiment(name, model, train_loader, val_loader, criterion, optimizer,
                   num_epochs, device=DEVICE, scheduler=None, freeze_bn_fn=None,
                   writer=None):
    """Run a full training experiment. Returns results dict."""
    history = {k: [] for k in ['train_loss', 'train_acc', 'train_f1',
                                'val_loss', 'val_acc', 'val_f1_macro', 'val_f1_micro']}
    best_f1 = -1.0
    best_state = None

    for epoch in range(num_epochs):
        # Optionally freeze BN during finetuning
        if freeze_bn_fn:
            model.train()
            freeze_bn_fn(model)

        tr_loss, tr_acc, tr_f1 = train_one_epoch(model, train_loader, criterion,
                                                  optimizer, device)
        va_loss, va_acc, va_f1_mac, va_f1_mic, _, _ = evaluate(model, val_loader,
                                                                criterion, device)
        if scheduler:
            scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['train_f1'].append(tr_f1)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        history['val_f1_macro'].append(va_f1_mac)
        history['val_f1_micro'].append(va_f1_mic)

        # TensorBoard logging
        if writer:
            writer.add_scalar("Loss/train", tr_loss, epoch)
            writer.add_scalar("Loss/val", va_loss, epoch)
            writer.add_scalar("Acc/train", tr_acc, epoch)
            writer.add_scalar("Acc/val", va_acc, epoch)
            writer.add_scalar("F1_macro/val", va_f1_mac, epoch)
            writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        if va_f1_mac > best_f1:
            best_f1 = va_f1_mac
            best_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{name}] Epoch {epoch+1:02d}/{num_epochs} | "
                  f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} | "
                  f"va_loss={va_loss:.4f} va_acc={va_acc:.3f} va_F1={va_f1_mac:.3f}")

    # Load best model and do final evaluation
    model.load_state_dict(best_state)
    va_loss, va_acc, va_f1_mac, va_f1_mic, preds, labels = evaluate(
        model, val_loader, criterion, device)

    print(f"  [{name}] Best: acc={va_acc:.4f} F1_macro={va_f1_mac:.4f} F1_micro={va_f1_mic:.4f}")

    if writer:
        writer.close()

    return {
        'history': history,
        'accuracy': va_acc,
        'f1_macro': va_f1_mac,
        'f1_micro': va_f1_mic,
        'val_loss': va_loss,
        'predictions': preds,
        'labels': labels,
        'model_state': best_state,
    }


## Section 7: Visualization Helpers
Training curves, confusion matrices, and formatted ablation tables.

In [ ]:
def plot_training_curves(history, title, save_path):
    """Plot train/val loss and accuracy curves."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    epochs = range(1, len(history['train_loss']) + 1)

    axes[0].plot(epochs, history['train_loss'], label='Train')
    axes[0].plot(epochs, history['val_loss'], label='Val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='Train')
    axes[1].plot(epochs, history['val_acc'], label='Val')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, history['train_f1'], label='Train F1')
    axes[2].plot(epochs, history['val_f1_macro'], label='Val F1-macro')
    axes[2].set_title('F1 Score')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_confusion_matrix(labels, preds, class_names, title, save_path):
    """Plot and save confusion matrix."""
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(8, 7))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', values_format='d', xticks_rotation=45)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def print_ablation_table(table_num, caption, rows, col_headers):
    """Print a formatted ablation table.

    rows: list of dicts with keys matching col_headers
    col_headers: list of (name, direction) tuples, e.g. ('Accuracy', 'up')
    """
    print(f"\n{'='*70}")
    print(f"Tab. {table_num} - {caption}")
    print(f"{'='*70}")

    # Find best and second-best for each metric
    header_line = f"{'Configuration':<35}"
    for name, direction in col_headers:
        arrow = '(^)' if direction == 'up' else '(v)'
        header_line += f" | {name} {arrow:>5}"
    print(header_line)
    print("-" * len(header_line))

    for col_name, direction in col_headers:
        vals = [r[col_name] for r in rows]
        if direction == 'up':
            sorted_vals = sorted(vals, reverse=True)
        else:
            sorted_vals = sorted(vals)
        for r in rows:
            if r[col_name] == sorted_vals[0]:
                r[f'{col_name}_rank'] = 'best'
            elif len(sorted_vals) > 1 and r[col_name] == sorted_vals[1]:
                r[f'{col_name}_rank'] = 'second'
            else:
                r[f'{col_name}_rank'] = ''

    for r in rows:
        line = f"{r['config']:<35}"
        for col_name, _ in col_headers:
            val = r[col_name]
            rank = r.get(f'{col_name}_rank', '')
            if rank == 'best':
                line += f" | **{val:.4f}**"
            elif rank == 'second':
                line += f" |  _{val:.4f}_ "
            else:
                line += f" |   {val:.4f}  "
        print(line)

    print(f"{'='*70}")
    print("** = best, _ = runner-up\n")


## Section 8: Data Loading Helpers
Factory functions for creating train/val DataLoaders for both datasets.

In [ ]:
def create_skin_loaders(img_size, aug_config='none', batch_size=CNN_BS,
                        for_pretrained=False, use_sampler=False):
    """Create train/val data loaders for skin dataset."""
    train_tf, val_tf = get_transforms('skin', img_size, aug_config, for_pretrained)
    train_ds = SkinDataset(
        os.path.join(SKIN_DIR, 'train.csv'),
        os.path.join(SKIN_DIR, 'train'), train_tf)
    val_ds = SkinDataset(
        os.path.join(SKIN_DIR, 'test.csv'),
        os.path.join(SKIN_DIR, 'test'), val_tf)

    sampler = get_weighted_sampler(train_ds.get_labels()) if use_sampler else None
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=(sampler is None), sampler=sampler,
                              num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, train_ds


def create_face_loaders(img_size, aug_config='none', batch_size=CNN_BS,
                        for_pretrained=False, use_sampler=False):
    """Create train/val data loaders for face dataset."""
    train_tf, val_tf = get_transforms('face', img_size, aug_config, for_pretrained)
    train_ds = FaceDataset(
        os.path.join(FACE_DIR, 'splits', 'train.csv'), FACE_DIR, train_tf)
    val_ds = FaceDataset(
        os.path.join(FACE_DIR, 'splits', 'local_test.csv'), FACE_DIR, val_tf)

    sampler = get_weighted_sampler(train_ds.get_labels()) if use_sampler else None
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=(sampler is None), sampler=sampler,
                              num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, train_ds


## Section 9: Augmentation Visualization (Req 3.1) [1p]
Visual examples of augmentations applied per class for both datasets.

In [ ]:
def visualize_augmentations_for_dataset(dataset_name):
    """Visualize augmentations for one example per class."""
    print(f"\n=== Augmentation Visualization: {dataset_name} ===")
    if dataset_name == 'skin':
        mean, std = SKIN_MEAN, SKIN_STD
        class_names = SKIN_CLASSES
        csv_path = os.path.join(SKIN_DIR, 'train.csv')
        img_dir = os.path.join(SKIN_DIR, 'train')
        base_ds = SkinDataset(csv_path, img_dir, transform=None)
    else:
        mean, std = FACE_MEAN, FACE_STD
        class_names = FACE_CLASSES
        csv_path = os.path.join(FACE_DIR, 'splits', 'train.csv')
        base_ds = FaceDataset(csv_path, FACE_DIR, transform=None)

    # Get one example per class
    class_examples = {}
    for idx in range(len(base_ds)):
        img, label = base_ds[idx]
        if label not in class_examples:
            class_examples[label] = img  # PIL image
        if len(class_examples) == len(class_names):
            break

    img_size = CNN_IMG_SIZE

    # Define augmentation transforms (PIL -> PIL, without Normalize/ToTensor)
    aug_pil = {
        'Original': transforms.Resize((img_size, img_size)),
        'H-Flip': transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=1.0),
        ]),
        'Rotation': transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomRotation(180 if dataset_name == 'skin' else 15),
        ]),
        'ColorJitter': transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        ]),
        'Crop+Resize': transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        ]),
        'All Combined': transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(180 if dataset_name == 'skin' else 15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        ]),
    }

    if dataset_name == 'skin':
        aug_pil['V-Flip'] = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomVerticalFlip(p=1.0),
        ])

    aug_names = list(aug_pil.keys())
    n_classes = len(class_names)
    n_aug = len(aug_names)

    fig, axes = plt.subplots(n_classes, n_aug, figsize=(2.5 * n_aug, 2.5 * n_classes))

    for row, cls_idx in enumerate(sorted(class_examples.keys())):
        pil_img = class_examples[cls_idx]

        for col, aug_name in enumerate(aug_names):
            aug_img = aug_pil[aug_name](pil_img)
            axes[row, col].imshow(np.array(aug_img))
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(aug_name, fontsize=9)
            if col == 0:
                axes[row, col].set_ylabel(class_names[cls_idx], fontsize=9, rotation=0,
                                          labelpad=60, va='center')

    title = f"Data Augmentation Visualization - {'Skin Moles' if dataset_name == 'skin' else 'Facial Expressions'}"
    fig.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    save_path = os.path.join(FIG_DIR, f"augmentations_{dataset_name}.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    # Print justification
    print(f"\n  Augmentation justification for {dataset_name}:")
    if dataset_name == 'skin':
        print("  - H/V Flip: dermoscopic images have no natural orientation -> all flips valid")
        print("  - Rotation (180 deg): moles can appear at any angle")
        print("  - ColorJitter (subtle): slight brightness/contrast variations simulate")
        print("    different imaging conditions; kept mild to preserve diagnostic color info")
        print("  - RandomCrop+Resize: simulates different zoom levels and positions")
    else:
        print("  - H-Flip: faces are approximately symmetric, reasonable augmentation")
        print("  - Rotation (15 deg): small head tilts are natural")
        print("  - NO Vertical Flip: faces do not appear upside down")
        print("  - ColorJitter: simulates different lighting conditions")
        print("  - RandomCrop+Resize: simulates different framing/zoom")


def run_augmentation_visualization():
    visualize_augmentations_for_dataset('skin')
    visualize_augmentations_for_dataset('face')


## Section 10: MLP Experiments (Req 3.2 + Ablation 3.5.1) [1.5p + 0.5p]
MLP training with/without dropout on both datasets.

In [ ]:
def run_mlp_ablation():
    """Train MLP with and without dropout on both datasets."""
    print("\n" + "="*70)
    print("MLP EXPERIMENTS: Dropout Ablation")
    print("="*70)

    all_results = {}
    table_num = 1

    for ds_name, create_fn, class_names in [
        ('skin', create_skin_loaders, SKIN_CLASSES),
        ('face', create_face_loaders, FACE_CLASSES),
    ]:
        print(f"\n--- Dataset: {ds_name} ---")
        input_dim = 3 * MLP_IMG_SIZE * MLP_IMG_SIZE
        num_classes = len(class_names)
        rows = []

        for use_dropout in [True, False]:
            tag = f"mlp_{'dropout' if use_dropout else 'no_dropout'}"
            exp_name = f"{ds_name}_{tag}"
            set_seed()

            train_loader, val_loader, _ = create_fn(
                MLP_IMG_SIZE, aug_config='none', batch_size=MLP_BS)

            model = MLP(input_dim, num_classes, use_dropout=use_dropout).to(DEVICE)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer, T_max=MLP_EPOCHS)
            writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, exp_name))

            print(f"\n  Training: {exp_name}")
            results = run_experiment(
                exp_name, model, train_loader, val_loader, criterion, optimizer,
                MLP_EPOCHS, scheduler=scheduler, writer=writer)

            all_results[exp_name] = results

            # Plot training curves
            plot_training_curves(
                results['history'],
                f"MLP {'+ Dropout' if use_dropout else '- Dropout'} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_curves.png"))

            # Plot confusion matrix
            plot_confusion_matrix(
                results['labels'], results['predictions'], class_names,
                f"MLP {'+ Dropout' if use_dropout else '- Dropout'} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_cm.png"))

            config_label = f"MLP {'+ Dropout (p=0.3)' if use_dropout else '- Dropout'}"
            rows.append({
                'config': config_label,
                'Accuracy': results['accuracy'],
                'F1-macro': results['f1_macro'],
                'F1-micro': results['f1_micro'],
                'Val Loss': results['val_loss'],
            })

        # Print ablation table
        ds_label = 'Skin Moles' if ds_name == 'skin' else 'Facial Expressions'
        print_ablation_table(
            table_num,
            f"MLP Dropout Ablation - {ds_label}",
            rows,
            [('Accuracy', 'up'), ('F1-macro', 'up'), ('F1-micro', 'up'), ('Val Loss', 'down')]
        )

        # Overfit/underfit analysis
        dropout_h = all_results[f"{ds_name}_mlp_dropout"]['history']
        gap = dropout_h['train_acc'][-1] - dropout_h['val_acc'][-1]
        print(f"  Analysis ({ds_name} MLP + Dropout):")
        print(f"    Train acc: {dropout_h['train_acc'][-1]:.3f}, Val acc: {dropout_h['val_acc'][-1]:.3f}")
        print(f"    Generalization gap: {gap:.3f}")
        if gap > 0.15:
            print("    -> Overfitting detected: train >> val. Dropout helps but more regularization may be needed.")
        elif gap < 0.05:
            print("    -> Good generalization. Model is well-regularized.")
        else:
            print("    -> Mild overfitting. Dropout provides reasonable regularization.")

        table_num += 1

    return all_results


## Section 11: CNN Experiments (Req 3.3 + Ablation 3.5.2 + 3.5.3) [1.5p + 0.5p + 1p]
CNN with/without BatchNorm and augmentation ablation (none/geometric/color/all).

In [ ]:
def run_cnn_ablation():
    """CNN experiments: BatchNorm ablation + Augmentation ablation."""
    print("\n" + "="*70)
    print("CNN EXPERIMENTS: BatchNorm + Augmentation Ablation")
    print("="*70)

    all_results = {}
    table_num = 3

    for ds_name, create_fn, class_names in [
        ('skin', create_skin_loaders, SKIN_CLASSES),
        ('face', create_face_loaders, FACE_CLASSES),
    ]:
        print(f"\n--- Dataset: {ds_name} ---")
        num_classes = len(class_names)
        first_kernel = 5 if ds_name == 'skin' else 3
        ds_label = 'Skin Moles' if ds_name == 'skin' else 'Facial Expressions'

        # --- Ablation 1: BatchNorm ---
        print(f"\n  === BatchNorm Ablation ({ds_name}) ===")
        bn_rows = []
        for use_bn in [True, False]:
            tag = f"cnn_{'bn' if use_bn else 'no_bn'}"
            exp_name = f"{ds_name}_{tag}"
            set_seed()

            train_loader, val_loader, _ = create_fn(
                CNN_IMG_SIZE, aug_config='none', batch_size=CNN_BS)

            model = CNN(num_classes, first_kernel=first_kernel,
                        use_batchnorm=use_bn).to(DEVICE)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer, T_max=CNN_EPOCHS)
            writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, exp_name))

            print(f"\n  Training: {exp_name}")
            results = run_experiment(
                exp_name, model, train_loader, val_loader, criterion, optimizer,
                CNN_EPOCHS, scheduler=scheduler, writer=writer)

            all_results[exp_name] = results
            plot_training_curves(
                results['history'],
                f"CNN {'+ BN' if use_bn else '- BN'} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_curves.png"))
            plot_confusion_matrix(
                results['labels'], results['predictions'], class_names,
                f"CNN {'+ BN' if use_bn else '- BN'} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_cm.png"))

            bn_rows.append({
                'config': f"CNN {'+ BatchNorm' if use_bn else '- BatchNorm'}",
                'Accuracy': results['accuracy'],
                'F1-macro': results['f1_macro'],
                'F1-micro': results['f1_micro'],
                'Val Loss': results['val_loss'],
            })

        print_ablation_table(
            table_num, f"CNN BatchNorm Ablation - {ds_label}", bn_rows,
            [('Accuracy', 'up'), ('F1-macro', 'up'), ('F1-micro', 'up'), ('Val Loss', 'down')])

        # Observation
        bn_gap = all_results[f"{ds_name}_cnn_bn"]['history']['train_acc'][-1] - \
                 all_results[f"{ds_name}_cnn_bn"]['history']['val_acc'][-1]
        nobn_gap = all_results[f"{ds_name}_cnn_no_bn"]['history']['train_acc'][-1] - \
                   all_results[f"{ds_name}_cnn_no_bn"]['history']['val_acc'][-1]
        print(f"  Observation: BN gen. gap = {bn_gap:.3f}, no-BN gen. gap = {nobn_gap:.3f}")
        print(f"  -> BatchNorm stabilizes training and typically improves generalization.")

        table_num += 1

        # --- Ablation 2: Augmentation (fix BN=True) ---
        print(f"\n  === Augmentation Ablation ({ds_name}, with BN) ===")
        aug_rows = []

        for aug_config in ['none', 'geometric', 'color', 'all']:
            tag = f"cnn_aug_{aug_config}"
            exp_name = f"{ds_name}_{tag}"
            set_seed()

            train_loader, val_loader, _ = create_fn(
                CNN_IMG_SIZE, aug_config=aug_config, batch_size=CNN_BS)

            model = CNN(num_classes, first_kernel=first_kernel,
                        use_batchnorm=True).to(DEVICE)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer, T_max=CNN_EPOCHS)
            writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, exp_name))

            print(f"\n  Training: {exp_name}")
            results = run_experiment(
                exp_name, model, train_loader, val_loader, criterion, optimizer,
                CNN_EPOCHS, scheduler=scheduler, writer=writer)

            all_results[exp_name] = results
            plot_training_curves(
                results['history'],
                f"CNN + BN, aug={aug_config} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_curves.png"))

            aug_label = {'none': 'No Augmentation', 'geometric': 'Geometric Only',
                         'color': 'Color Only', 'all': 'All Augmentations'}[aug_config]
            aug_rows.append({
                'config': aug_label,
                'Accuracy': results['accuracy'],
                'F1-macro': results['f1_macro'],
                'F1-micro': results['f1_micro'],
                'Val Loss': results['val_loss'],
            })

        print_ablation_table(
            table_num, f"CNN Augmentation Ablation - {ds_label}", aug_rows,
            [('Accuracy', 'up'), ('F1-macro', 'up'), ('F1-micro', 'up'), ('Val Loss', 'down')])

        # Find best augmentation
        best_aug = 'all'
        best_f1 = -1
        for ac in ['none', 'geometric', 'color', 'all']:
            f1 = all_results[f"{ds_name}_cnn_aug_{ac}"]['f1_macro']
            if f1 > best_f1:
                best_f1 = f1
                best_aug = ac
        all_results[f"{ds_name}_best_aug"] = best_aug
        print(f"  -> Best augmentation for {ds_name}: {best_aug} (F1-macro={best_f1:.4f})")

        # Observation about train/val gap reduction with augmentation
        no_aug_gap = all_results[f"{ds_name}_cnn_aug_none"]['history']['train_acc'][-1] - \
                     all_results[f"{ds_name}_cnn_aug_none"]['history']['val_acc'][-1]
        all_aug_gap = all_results[f"{ds_name}_cnn_aug_all"]['history']['train_acc'][-1] - \
                      all_results[f"{ds_name}_cnn_aug_all"]['history']['val_acc'][-1]
        print(f"  Gen. gap: no aug = {no_aug_gap:.3f}, all aug = {all_aug_gap:.3f}")
        print(f"  -> Augmentation reduces overfitting by narrowing the train/val gap.")

        table_num += 1

    return all_results


## Section 12: Class Imbalance Ablation (Req 3.5.4) [1p]
No handling vs Weighted CE vs Weighted Sampler.

In [ ]:
def run_imbalance_ablation(cnn_results=None):
    """Train CNN with different class imbalance handling strategies."""
    print("\n" + "="*70)
    print("CLASS IMBALANCE ABLATION")
    print("="*70)

    all_results = {}
    table_num = 7

    for ds_name, create_fn, class_names in [
        ('skin', create_skin_loaders, SKIN_CLASSES),
        ('face', create_face_loaders, FACE_CLASSES),
    ]:
        print(f"\n--- Dataset: {ds_name} ---")
        num_classes = len(class_names)
        first_kernel = 5 if ds_name == 'skin' else 3
        ds_label = 'Skin Moles' if ds_name == 'skin' else 'Facial Expressions'

        # Use best augmentation from previous step if available
        best_aug = 'all'
        if cnn_results and f"{ds_name}_best_aug" in cnn_results:
            best_aug = cnn_results[f"{ds_name}_best_aug"]

        # Print class distribution
        train_loader_tmp, _, train_ds_tmp = create_fn(
            CNN_IMG_SIZE, aug_config=best_aug, batch_size=CNN_BS)
        labels = train_ds_tmp.get_labels()
        dist = Counter(labels.tolist())
        print(f"  Class distribution: {dict(sorted(dist.items()))}")

        rows = []

        # Strategy 1: No handling (baseline)
        for strategy in ['none', 'weighted_ce', 'weighted_sampler']:
            tag = f"cnn_imb_{strategy}"
            exp_name = f"{ds_name}_{tag}"
            set_seed()

            use_sampler = (strategy == 'weighted_sampler')
            train_loader, val_loader, train_ds = create_fn(
                CNN_IMG_SIZE, aug_config=best_aug, batch_size=CNN_BS,
                use_sampler=use_sampler)

            model = CNN(num_classes, first_kernel=first_kernel,
                        use_batchnorm=True).to(DEVICE)

            if strategy == 'weighted_ce':
                weights = get_class_weights(train_ds.get_labels(), num_classes, DEVICE)
                criterion = nn.CrossEntropyLoss(weight=weights)
            elif strategy == 'weighted_sampler':
                criterion = nn.CrossEntropyLoss()
            else:
                criterion = nn.CrossEntropyLoss()

            optimizer = optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer, T_max=CNN_EPOCHS)
            writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, exp_name))

            print(f"\n  Training: {exp_name}")
            results = run_experiment(
                exp_name, model, train_loader, val_loader, criterion, optimizer,
                CNN_EPOCHS, scheduler=scheduler, writer=writer)

            all_results[exp_name] = results
            plot_training_curves(
                results['history'],
                f"CNN imbalance={strategy} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_curves.png"))
            plot_confusion_matrix(
                results['labels'], results['predictions'], class_names,
                f"CNN imbalance={strategy} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_cm.png"))

            strat_label = {'none': 'No Handling (baseline)',
                          'weighted_ce': 'Weighted CrossEntropy',
                          'weighted_sampler': 'Weighted Sampler'}[strategy]
            rows.append({
                'config': strat_label,
                'Accuracy': results['accuracy'],
                'F1-macro': results['f1_macro'],
                'F1-micro': results['f1_micro'],
                'Val Loss': results['val_loss'],
            })

        print_ablation_table(
            table_num, f"Class Imbalance Ablation - {ds_label}", rows,
            [('Accuracy', 'up'), ('F1-macro', 'up'), ('F1-micro', 'up'), ('Val Loss', 'down')])

        # Observation
        baseline_f1 = all_results[f"{ds_name}_cnn_imb_none"]['f1_macro']
        wce_f1 = all_results[f"{ds_name}_cnn_imb_weighted_ce"]['f1_macro']
        ws_f1 = all_results[f"{ds_name}_cnn_imb_weighted_sampler"]['f1_macro']
        print(f"  F1-macro: baseline={baseline_f1:.4f}, weighted CE={wce_f1:.4f}, "
              f"weighted sampler={ws_f1:.4f}")
        if max(wce_f1, ws_f1) > baseline_f1:
            print("  -> Class imbalance handling improves F1-macro on minority classes.")
        else:
            print("  -> Imbalance handling did not improve overall F1; dataset may be less skewed than expected.")

        table_num += 1

    return all_results


## Section 13: Fine-Tuning (Req 3.4 + Ablation 3.5.5) [2p + 1p]
ResNet-18 fine-tuning with differential LR and warmup ablation.

In [ ]:
def create_finetune_model(num_classes):
    """Create a ResNet-18 model for fine-tuning with trainable backbone."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Replace classification head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )

    return model


def get_finetune_param_groups(model, lr_head, lr_backbone):
    """Return parameter groups with differential learning rates."""
    head_params = list(model.fc.parameters())
    head_ids = set(id(p) for p in head_params)
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    return [
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params, 'lr': lr_head},
    ]


def run_finetune_ablation(cnn_results=None):
    """Fine-tuning ResNet-18 with differential LR (backbone vs head) and warmup ablation."""
    print("\n" + "="*70)
    print("FINE-TUNING EXPERIMENTS: LR Warmup Ablation")
    print("="*70)

    all_results = {}
    table_num = 9

    for ds_name, create_fn, class_names in [
        ('skin', create_skin_loaders, SKIN_CLASSES),
        ('face', create_face_loaders, FACE_CLASSES),
    ]:
        print(f"\n--- Dataset: {ds_name} ---")
        num_classes = len(class_names)
        ds_label = 'Skin Moles' if ds_name == 'skin' else 'Facial Expressions'

        # Use best augmentation
        best_aug = 'all'
        if cnn_results and f"{ds_name}_best_aug" in cnn_results:
            best_aug = cnn_results[f"{ds_name}_best_aug"]

        rows = []

        for use_warmup in [False, True]:
            tag = f"ft_{'warmup' if use_warmup else 'no_warmup'}"
            exp_name = f"{ds_name}_{tag}"
            set_seed()

            train_loader, val_loader, train_ds = create_fn(
                FT_IMG_SIZE, aug_config=best_aug, batch_size=FT_BS,
                for_pretrained=True, use_sampler=False)

            model = create_finetune_model(num_classes).to(DEVICE)

            # Use weighted CE for class imbalance (no sampler to avoid double weighting)
            weights = get_class_weights(train_ds.get_labels(), num_classes, DEVICE)
            criterion = nn.CrossEntropyLoss(weight=weights)

            # Differential LR: lower for backbone, higher for head
            param_groups = get_finetune_param_groups(model, FT_LR_HEAD, FT_LR_BACKBONE)
            optimizer = optim.Adam(param_groups, weight_decay=WEIGHT_DECAY)

            if use_warmup:
                warmup_epochs = 3
                warmup_sched = LambdaLR(optimizer,
                                        lr_lambda=lambda e: (e + 1) / warmup_epochs)
                cosine_sched = CosineAnnealingLR(optimizer,
                                                 T_max=FT_EPOCHS - warmup_epochs)
                scheduler = SequentialLR(optimizer,
                                        schedulers=[warmup_sched, cosine_sched],
                                        milestones=[warmup_epochs])
            else:
                scheduler = CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)

            writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, exp_name))

            print(f"\n  Training: {exp_name}")
            results = run_experiment(
                exp_name, model, train_loader, val_loader, criterion, optimizer,
                FT_EPOCHS, scheduler=scheduler, writer=writer)

            all_results[exp_name] = results
            plot_training_curves(
                results['history'],
                f"ResNet-18 FT {'+ Warmup' if use_warmup else '- Warmup'} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_curves.png"))
            plot_confusion_matrix(
                results['labels'], results['predictions'], class_names,
                f"ResNet-18 FT {'+ Warmup' if use_warmup else '- Warmup'} on {ds_name}",
                os.path.join(FIG_DIR, f"{exp_name}_cm.png"))

            warmup_label = "Full FT + Warmup (3ep)" if use_warmup else "Full FT, No Warmup"
            rows.append({
                'config': warmup_label,
                'Accuracy': results['accuracy'],
                'F1-macro': results['f1_macro'],
                'F1-micro': results['f1_micro'],
                'Val Loss': results['val_loss'],
            })

        print_ablation_table(
            table_num, f"Fine-tuning LR Warmup Ablation - {ds_label}", rows,
            [('Accuracy', 'up'), ('F1-macro', 'up'), ('F1-micro', 'up'), ('Val Loss', 'down')])

        # Observation
        nw = all_results[f"{ds_name}_ft_no_warmup"]
        w = all_results[f"{ds_name}_ft_warmup"]
        print(f"  No warmup: F1={nw['f1_macro']:.4f}, Warmup: F1={w['f1_macro']:.4f}")
        # Check early instability
        nw_early_loss = np.mean(nw['history']['val_loss'][:3])
        w_early_loss = np.mean(w['history']['val_loss'][:3])
        print(f"  Avg val loss (first 3 epochs): no_warmup={nw_early_loss:.4f}, warmup={w_early_loss:.4f}")
        if w_early_loss < nw_early_loss:
            print("  -> Warmup reduces early instability, leading to smoother convergence.")
        else:
            print("  -> Warmup did not significantly impact early training in this case.")

        table_num += 1

    return all_results


## Section 14: Curriculum Learning (BONUS) [2p]
Self-paced curriculum learning vs standard training.

In [ ]:
def compute_per_sample_loss(model, dataset, device, batch_size=128):
    """Compute loss for each sample using current model."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS)
    losses = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            losses.append(loss.cpu())
    return torch.cat(losses)


def pacing_function(epoch, total_epochs, start_frac=0.3):
    """Linear pacing: start with start_frac of data, reach 100% at half training."""
    return min(1.0, start_frac + (1 - start_frac) * (epoch / (total_epochs * 0.5)))


def run_curriculum_experiment(name, model, full_train_ds, val_loader, criterion,
                              optimizer, num_epochs, device=DEVICE, scheduler=None):
    """Self-paced curriculum learning experiment."""
    history = {k: [] for k in ['train_loss', 'train_acc', 'train_f1',
                                'val_loss', 'val_acc', 'val_f1_macro', 'val_f1_micro']}
    best_f1 = -1.0
    best_state = None
    writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, name))

    for epoch in range(num_epochs):
        # Compute per-sample losses to determine difficulty
        losses = compute_per_sample_loss(model, full_train_ds, device)

        # Determine how much data to use this epoch
        frac = pacing_function(epoch, num_epochs)
        threshold = torch.quantile(losses, q=min(1.0, frac))
        easy_idx = (losses <= threshold).nonzero(as_tuple=True)[0].tolist()

        # Create subset loader for this epoch
        subset = Subset(full_train_ds, easy_idx)
        train_loader = DataLoader(subset, batch_size=CNN_BS, shuffle=True,
                                  num_workers=NUM_WORKERS, drop_last=True)

        # Train
        model.train()
        tr_loss, tr_acc, tr_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, device)

        # Evaluate on FULL val set
        va_loss, va_acc, va_f1_mac, va_f1_mic, _, _ = evaluate(
            model, val_loader, criterion, device)

        if scheduler:
            scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['train_f1'].append(tr_f1)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        history['val_f1_macro'].append(va_f1_mac)
        history['val_f1_micro'].append(va_f1_mic)

        writer.add_scalar("Loss/train", tr_loss, epoch)
        writer.add_scalar("Loss/val", va_loss, epoch)
        writer.add_scalar("Acc/val", va_acc, epoch)
        writer.add_scalar("F1_macro/val", va_f1_mac, epoch)
        writer.add_scalar("Curriculum/frac", frac, epoch)
        writer.add_scalar("Curriculum/n_samples", len(easy_idx), epoch)

        if va_f1_mac > best_f1:
            best_f1 = va_f1_mac
            best_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{name}] Epoch {epoch+1:02d}/{num_epochs} | "
                  f"frac={frac:.2f} n={len(easy_idx)} | "
                  f"va_acc={va_acc:.3f} va_F1={va_f1_mac:.3f}")

    model.load_state_dict(best_state)
    va_loss, va_acc, va_f1_mac, va_f1_mic, preds, labels = evaluate(
        model, val_loader, criterion, device)

    print(f"  [{name}] Best: acc={va_acc:.4f} F1_macro={va_f1_mac:.4f}")
    writer.close()

    return {
        'history': history,
        'accuracy': va_acc,
        'f1_macro': va_f1_mac,
        'f1_micro': va_f1_mic,
        'val_loss': va_loss,
        'predictions': preds,
        'labels': labels,
        'model_state': best_state,
    }


def run_curriculum_learning():
    """Compare standard training vs curriculum learning."""
    print("\n" + "="*70)
    print("BONUS: CURRICULUM LEARNING")
    print("="*70)

    all_results = {}
    table_num = 11

    for ds_name, create_fn, class_names in [
        ('skin', create_skin_loaders, SKIN_CLASSES),
        ('face', create_face_loaders, FACE_CLASSES),
    ]:
        print(f"\n--- Dataset: {ds_name} ---")
        num_classes = len(class_names)
        first_kernel = 5 if ds_name == 'skin' else 3
        ds_label = 'Skin Moles' if ds_name == 'skin' else 'Facial Expressions'
        rows = []

        # Baseline: standard training (CNN + BN + all aug)
        set_seed()
        train_loader, val_loader, train_ds = create_fn(
            CNN_IMG_SIZE, aug_config='all', batch_size=CNN_BS)

        exp_name = f"{ds_name}_cnn_standard"
        model = CNN(num_classes, first_kernel=first_kernel, use_batchnorm=True).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=CNN_EPOCHS)

        print(f"\n  Training baseline: {exp_name}")
        results = run_experiment(
            exp_name, model, train_loader, val_loader, criterion, optimizer,
            CNN_EPOCHS, scheduler=scheduler)
        all_results[exp_name] = results
        rows.append({
            'config': 'Standard Training (baseline)',
            'Accuracy': results['accuracy'],
            'F1-macro': results['f1_macro'],
            'F1-micro': results['f1_micro'],
            'Val Loss': results['val_loss'],
        })

        # Curriculum: self-paced learning
        set_seed()
        # Need a dataset with the same transforms for curriculum (no shuffle in DS)
        train_tf, val_tf = get_transforms(ds_name, CNN_IMG_SIZE, aug_config='all')
        if ds_name == 'skin':
            full_train_ds = SkinDataset(
                os.path.join(SKIN_DIR, 'train.csv'),
                os.path.join(SKIN_DIR, 'train'), train_tf)
        else:
            full_train_ds = FaceDataset(
                os.path.join(FACE_DIR, 'splits', 'train.csv'), FACE_DIR, train_tf)

        exp_name = f"{ds_name}_cnn_curriculum"
        model = CNN(num_classes, first_kernel=first_kernel, use_batchnorm=True).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=CNN_EPOCHS)

        print(f"\n  Training curriculum: {exp_name}")
        results = run_curriculum_experiment(
            exp_name, model, full_train_ds, val_loader, criterion, optimizer,
            CNN_EPOCHS, scheduler=scheduler)
        all_results[exp_name] = results

        plot_training_curves(
            results['history'],
            f"Curriculum Learning on {ds_name}",
            os.path.join(FIG_DIR, f"{exp_name}_curves.png"))

        rows.append({
            'config': 'Self-Paced Curriculum',
            'Accuracy': results['accuracy'],
            'F1-macro': results['f1_macro'],
            'F1-micro': results['f1_micro'],
            'Val Loss': results['val_loss'],
        })

        print_ablation_table(
            table_num, f"Curriculum Learning Ablation - {ds_label}", rows,
            [('Accuracy', 'up'), ('F1-macro', 'up'), ('F1-micro', 'up'), ('Val Loss', 'down')])

        std_f1 = all_results[f"{ds_name}_cnn_standard"]['f1_macro']
        cur_f1 = all_results[f"{ds_name}_cnn_curriculum"]['f1_macro']
        print(f"  Standard F1={std_f1:.4f}, Curriculum F1={cur_f1:.4f}")
        if cur_f1 > std_f1:
            print("  -> Curriculum learning improves performance by focusing on harder examples later.")
        else:
            print("  -> Curriculum learning did not improve over standard training here; "
                  "the difficulty proxy may need refinement.")

        table_num += 1

    return all_results


## Section 15: Kaggle Submissions
Generate predictions using the best model per dataset.

In [ ]:
@torch.no_grad()
def generate_predictions(model, loader, device):
    """Generate predictions for a test loader."""
    model.eval()
    all_preds = []
    all_ids = []
    for batch in loader:
        images, ids = batch
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_ids.extend(ids)
    return all_ids, all_preds


def generate_kaggle_submissions(all_results_dict):
    """Generate Kaggle submission files using the best models overall."""
    print("\n" + "="*70)
    print("GENERATING KAGGLE SUBMISSIONS")
    print("="*70)

    # Find the best model for each dataset by F1-macro across ALL experiments
    def find_best(ds_prefix, results_dict):
        best_key, best_f1 = None, -1
        for key, res in results_dict.items():
            if key.startswith(ds_prefix) and 'f1_macro' in res and 'model_state' in res:
                if res['f1_macro'] > best_f1:
                    best_f1 = res['f1_macro']
                    best_key = key
        return best_key, best_f1

    # --- Skin Moles ---
    print("\n  Generating skin moles submission...")
    skin_key, skin_f1 = find_best('skin_', all_results_dict)
    print(f"  Best model: {skin_key} (F1-macro={skin_f1:.4f})")

    # Determine if the best model is a fine-tuned (ResNet) or CNN model
    is_ft = 'ft_' in skin_key
    img_size = FT_IMG_SIZE if is_ft else CNN_IMG_SIZE
    for_pretrained = is_ft
    _, val_tf = get_transforms('skin', img_size, aug_config='none', for_pretrained=for_pretrained)
    test_ds = SkinTestDataset(
        os.path.join(SKIN_DIR, 'kaggle_test.csv'),
        os.path.join(SKIN_DIR, 'kaggle_test'), val_tf)
    test_loader = DataLoader(test_ds, batch_size=FT_BS, shuffle=False,
                             num_workers=NUM_WORKERS)

    if is_ft:
        model = create_finetune_model(len(SKIN_CLASSES)).to(DEVICE)
    else:
        model = CNN(len(SKIN_CLASSES)).to(DEVICE)
    model.load_state_dict(all_results_dict[skin_key]['model_state'])

    ids, preds = generate_predictions(model, test_loader, DEVICE)
    idx_to_class = {i: c for i, c in enumerate(SKIN_CLASSES)}

    skin_sub_path = os.path.join(SUB_DIR, 'skin_submission.csv')
    with open(skin_sub_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['imagine', 'diagnostic'])
        for img_name, pred in zip(ids, preds):
            writer.writerow([img_name, idx_to_class[pred]])
    print(f"  Saved: {skin_sub_path} ({len(ids)} predictions)")

    # --- Facial Expressions ---
    print("\n  Generating facial expressions submission...")
    face_key, face_f1 = find_best('face_', all_results_dict)
    print(f"  Best model: {face_key} (F1-macro={face_f1:.4f})")

    is_ft = 'ft_' in face_key
    img_size = FT_IMG_SIZE if is_ft else CNN_IMG_SIZE
    for_pretrained = is_ft
    _, val_tf = get_transforms('face', img_size, aug_config='none', for_pretrained=for_pretrained)
    test_ds = FaceRemoteTestDataset(
        os.path.join(FACE_DIR, 'splits', 'remote_test.csv'), FACE_DIR, val_tf)
    test_loader = DataLoader(test_ds, batch_size=FT_BS, shuffle=False,
                             num_workers=NUM_WORKERS)

    if is_ft:
        model = create_finetune_model(len(FACE_CLASSES)).to(DEVICE)
    else:
        model = CNN(len(FACE_CLASSES)).to(DEVICE)
    model.load_state_dict(all_results_dict[face_key]['model_state'])

    ids, preds = generate_predictions(model, test_loader, DEVICE)

    face_sub_path = os.path.join(SUB_DIR, 'face_submission.csv')
    with open(face_sub_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'label'])
        for img_id, pred in zip(ids, preds):
            writer.writerow([img_id, pred + 1])  # Convert 0-6 back to 1-7
    print(f"  Saved: {face_sub_path} ({len(ids)} predictions)")


## Section 16: Summary Report
Comprehensive results summary across all experiments.

In [ ]:
def print_full_summary(mlp_res, cnn_res, imb_res, ft_res, cur_res):
    """Print a comprehensive summary of all experiments."""
    print("\n" + "="*70)
    print("FULL SUMMARY OF ALL EXPERIMENTS")
    print("="*70)

    summary = []
    for ds_name in ['skin', 'face']:
        ds_label = 'Skin' if ds_name == 'skin' else 'Face'

        entries = [
            (f'{ds_label} MLP + Dropout', mlp_res.get(f'{ds_name}_mlp_dropout', {})),
            (f'{ds_label} MLP - Dropout', mlp_res.get(f'{ds_name}_mlp_no_dropout', {})),
            (f'{ds_label} CNN + BN', cnn_res.get(f'{ds_name}_cnn_bn', {})),
            (f'{ds_label} CNN - BN', cnn_res.get(f'{ds_name}_cnn_no_bn', {})),
            (f'{ds_label} CNN no aug', cnn_res.get(f'{ds_name}_cnn_aug_none', {})),
            (f'{ds_label} CNN geo aug', cnn_res.get(f'{ds_name}_cnn_aug_geometric', {})),
            (f'{ds_label} CNN color aug', cnn_res.get(f'{ds_name}_cnn_aug_color', {})),
            (f'{ds_label} CNN all aug', cnn_res.get(f'{ds_name}_cnn_aug_all', {})),
            (f'{ds_label} CNN no imb', imb_res.get(f'{ds_name}_cnn_imb_none', {})),
            (f'{ds_label} CNN weighted CE', imb_res.get(f'{ds_name}_cnn_imb_weighted_ce', {})),
            (f'{ds_label} CNN w. sampler', imb_res.get(f'{ds_name}_cnn_imb_weighted_sampler', {})),
            (f'{ds_label} FT no warmup', ft_res.get(f'{ds_name}_ft_no_warmup', {})),
            (f'{ds_label} FT warmup', ft_res.get(f'{ds_name}_ft_warmup', {})),
            (f'{ds_label} Curriculum std', cur_res.get(f'{ds_name}_cnn_standard', {})),
            (f'{ds_label} Curriculum SPL', cur_res.get(f'{ds_name}_cnn_curriculum', {})),
        ]

        for label, res in entries:
            if res and 'accuracy' in res:
                summary.append({
                    'Experiment': label,
                    'Accuracy': f"{res['accuracy']:.4f}",
                    'F1-macro': f"{res['f1_macro']:.4f}",
                    'F1-micro': f"{res['f1_micro']:.4f}",
                })

    if summary:
        df = pd.DataFrame(summary)
        print(df.to_string(index=False))

    print("\n\nResults, figures, and submissions saved to:")
    print(f"  Figures:     {FIG_DIR}/")
    print(f"  Submissions: {SUB_DIR}/")
    print(f"  TensorBoard: {RUNS_DIR}/ (run: tensorboard --logdir runs)")


## Main Execution
Run all experiments sequentially.

In [ ]:
def main():
    print("="*70)
    print("TEMA 2 IA - NEURAL NETWORK IMAGE CLASSIFICATION")
    print(f"Device: {DEVICE}")
    print("="*70)

    # Req 3.1: Augmentation visualization
    print("\n\n[1/7] Augmentation Visualization (Req 3.1)")
    run_augmentation_visualization()

    # Req 3.2 + 3.5.1: MLP + Dropout ablation
    print("\n\n[2/7] MLP Experiments (Req 3.2 + 3.5.1)")
    mlp_results = run_mlp_ablation()

    # Req 3.3 + 3.5.2 + 3.5.3: CNN + BN ablation + Aug ablation
    print("\n\n[3/7] CNN Experiments (Req 3.3 + 3.5.2 + 3.5.3)")
    cnn_results = run_cnn_ablation()

    # Req 3.5.4: Class imbalance ablation
    print("\n\n[4/7] Class Imbalance Experiments (Req 3.5.4)")
    imb_results = run_imbalance_ablation(cnn_results)

    # Req 3.4 + 3.5.5: Fine-tuning + Warmup ablation
    print("\n\n[5/7] Fine-tuning Experiments (Req 3.4 + 3.5.5)")
    ft_results = run_finetune_ablation(cnn_results)

    # Bonus: Curriculum Learning
    print("\n\n[6/7] Curriculum Learning (Bonus)")
    cur_results = run_curriculum_learning()

    # Kaggle Submissions - merge all results and pick best model per dataset
    print("\n\n[7/7] Kaggle Submissions")
    all_results = {}
    for d in [mlp_results, cnn_results, imb_results, ft_results, cur_results]:
        all_results.update(d)
    generate_kaggle_submissions(all_results)

    # Summary
    print_full_summary(mlp_results, cnn_results, imb_results, ft_results, cur_results)

    print("\n\nDONE! All experiments completed successfully.")


if __name__ == "__main__":
    main()
